# Cross-soup re-blend — α по gray (без train)

Быстрый submit после `pipeline_public.ipynb`: **без v3 train**, только blend + TTA + zip.

**Зачем:** первый submit взял **α=0.05** (best offline composite → высокий problem, низкий gray) → LB **0.5527**.
Здесь фиксируем **α по gray** (default **0.15**, offline gray **~0.554**).

Формула: `(1−α)·v2_step2000 + α·v3_step400`, eval и submit **с symmetry TTA**.

| α | offline gray (TTA, fresh v2) | LB (если есть) |
|---|---------------------------|----------------|
| 0.05 | 0.5502 | **0.5527** ← прошлый submit |
| 0.10 | 0.5522 | ref 04 recipe |
| **0.15** | **0.5544** | **рекомендуется** |
| 0.18 | 0.5559 | max gray в grid |

In [ ]:
import json, os, subprocess, sys
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow
import sklearn
import torch, transformers

START = Path.cwd().resolve()
masked = lambda path: f"***/{Path(path).name}"

def resolve_script(env_name: str, default: str) -> Path:
    rel = os.getenv(env_name, default)
    p = Path(rel).expanduser()
    if p.is_absolute() and p.exists():
        return p.resolve()
    for base in [START, *START.parents]:
        cand = (base / rel).resolve()
        if cand.exists():
            return cand
    raise FileNotFoundError(f"Не найден {rel}")

def find_file(name: str, *roots: Path) -> Path:
    for root in roots:
        p = (root / name).resolve()
        if p.exists():
            return p
    raise FileNotFoundError(name)

BLEND_SCRIPT = resolve_script("STAGE_B_BLEND_SCRIPT", "final_4_models/scripts/blend_checkpoint_soup.py")
BUILD_SUBMIT = resolve_script("TTA_BUILD_SUBMIT", "notebooks/symmetry_tta_v3_soup/build_submit.py")
SUBMIT_TEMPLATE = resolve_script(
    "TTA_SUBMIT_TEMPLATE", "final_4_models/submit/matching-bge-human-ft/src/utils.py"
)

PROJECT_DIR = BLEND_SCRIPT.parent.parent.parent
LIB = find_file("pair_text_v1.py", START / "../lib", START.parent / "lib", PROJECT_DIR / "notebooks/lib")
sys.path.insert(0, str(LIB.parent))

assert BLEND_SCRIPT.exists() and BUILD_SUBMIT.exists()
assert "symmetry TTA" in SUBMIT_TEMPLATE.read_text() or "enc_rev" in SUBMIT_TEMPLATE.read_text()

EXPECTED_VERSIONS = {
    "python": "3.12", "torch": "2.6.0+cu124", "transformers": "4.57.6",
    "numpy": "2.2.6", "pandas": "2.3.3", "pyarrow": "23.0.1", "scikit-learn": "1.8.0",
}
ACTUAL = {
    "python": ".".join(map(str, sys.version_info[:2])),
    "torch": torch.__version__, "transformers": transformers.__version__,
    "numpy": np.__version__, "pandas": pd.__version__,
    "pyarrow": pyarrow.__version__, "scikit-learn": sklearn.__version__,
}
assert ACTUAL == EXPECTED_VERSIONS, ACTUAL

print("blend:", masked(BLEND_SCRIPT))
print("build_submit:", masked(BUILD_SUBMIT))
print("TTA template OK")

In [ ]:
# ── конфиг ──────────────────────────────────────────────────────────────
RUN_ID = "cross_tta_gray_a015"
GPU_ID = "0"                    # одна GPU достаточно (~5 мин на α)
MIN_FREE_MIB = 50_000

# fixed — один α для submit | gray_grid — мини-grid, выбор max gray_full
MODE = "fixed"
SUBMIT_ALPHA = 0.15             # gray-optimal (0.10=ref04, 0.18=max gray)
GRAY_GRID = "0.12,0.14,0.15,0.16,0.18"
MIN_PROBLEM_AP = 0.632          # фильтр при MODE=gray_grid

USE_REFERENCE_V2 = False        # True → reference v2@2000 (gray выше offline)
SYMMETRY_TTA_EVAL = True
RUN_SUBMIT = True

OUTPUT_ROOT = Path(os.getenv("TTA_OUTPUT_ROOT", START / "output")).expanduser().resolve()
PRIOR_RUN = Path(os.getenv(
    "CROSS_TTA_PRIOR_RUN",
    START / "output" / "cross_tta_s42_01",
)).expanduser().resolve()

CKPT_A = Path(os.getenv(
    "V3_CKPT_400",
    PRIOR_RUN / "stageb_v3/checkpoints/step_00400.pt",
)).resolve()

V2_FRESH = Path(os.getenv(
    "V2_CKPT_2000",
    START.parent / "v2_soup_lb_5522/output/v2_soup_s42_01/stageb_v2/checkpoints/step_02000.pt",
)).resolve()
V2_REF = Path(os.getenv(
    "V2_REF_CKPT_2000",
    PROJECT_DIR / "final_4_models/runs/01_v2_step2000/checkpoints/step_02000.pt",
)).resolve()
CKPT_B = V2_REF if USE_REFERENCE_V2 else V2_FRESH

SOUP_OUT = OUTPUT_ROOT / RUN_ID / "soup_run"
SOUP_OUT.mkdir(parents=True, exist_ok=True)

assert CKPT_A.exists(), f"Нет v3 step_400: ***/{CKPT_A.name} — сначала pipeline_public"
assert CKPT_B.exists(), f"Нет v2 step_2000: ***/{CKPT_B.name}"

CONFIG = {
    "mode": MODE,
    "submit_alpha": SUBMIT_ALPHA,
    "gray_grid": GRAY_GRID,
    "min_problem_ap": MIN_PROBLEM_AP,
    "use_reference_v2": USE_REFERENCE_V2,
    "v3_ckpt": f"***/{CKPT_A.name}",
    "v2_ckpt": f"***/{CKPT_B.name}",
    "soup_out": f"***/{SOUP_OUT.name}",
    "symmetry_tta": SYMMETRY_TTA_EVAL,
    "prior_lb_alpha005": 0.5526942615,
}
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

def run_cmd(cmd: str) -> int:
    shown = cmd
    for secret in (str(PROJECT_DIR), str(PRIOR_RUN), str(SOUP_OUT), str(CKPT_A), str(CKPT_B)):
        shown = shown.replace(secret, "***")
    print("$", shown, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    assert p.stdout is not None
    for line in p.stdout:
        print(line, end="", flush=True)
    return p.wait()

def gpu_free_mib(idx: int) -> int:
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=index,memory.used,memory.total", "--format=csv,noheader,nounits"],
        text=True,
    )
    for ln in out.strip().splitlines():
        i, used, total = (int(x.strip()) for x in ln.split(","))
        if i == idx:
            return total - used
    raise RuntimeError(f"GPU {idx} not found")

gpu_i = int(GPU_ID.split(",")[0])
free = gpu_free_mib(gpu_i)
print(f"GPU {gpu_i} free MiB: {free}")
assert free >= MIN_FREE_MIB, free

In [ ]:
from verify_pair_text import main as verify_main
assert verify_main() == 0

In [ ]:
scripts_dir = BLEND_SCRIPT.parent
sys.path.insert(0, str(scripts_dir))
sys.path.insert(0, str(LIB.parent))
from verify_pair_text import patch_score_ensemble_v1
patch_score_ensemble_v1()

tta_flag = "--symmetry-tta" if SYMMETRY_TTA_EVAL else ""
blend_base = (
    f"cd {PROJECT_DIR} && PYTHONPATH={scripts_dir}:{LIB.parent} "
    f"python {BLEND_SCRIPT} --ckpt-a {CKPT_A} --ckpt-b {CKPT_B} "
    f"--out-dir {SOUP_OUT} --skip-submit --gpu {GPU_ID} {tta_flag}"
)

if MODE == "fixed":
    chosen_alpha = SUBMIT_ALPHA
    assert run_cmd(f"{blend_base} --alphas {chosen_alpha}") == 0
elif MODE == "gray_grid":
    assert run_cmd(f"{blend_base} --alphas {GRAY_GRID}") == 0
    grid = pd.DataFrame(json.loads((SOUP_OUT / "soup_grid.json").read_text()))
    display(grid.round(4))
    ok = grid[grid.problem_ap >= MIN_PROBLEM_AP]
    if ok.empty:
        ok = grid
    chosen_alpha = float(ok.loc[ok.gray_full.idxmax(), "alpha_step400"])
    print(f"gray_grid pick: α={chosen_alpha} (max gray, problem≥{MIN_PROBLEM_AP})")
    if abs(chosen_alpha - json.loads((SOUP_OUT / "metrics.json").read_text())["blend"]["best_alpha"]) > 1e-6:
        print(f"re-export α={chosen_alpha} (blend script выбрал composite-best)")
        assert run_cmd(f"{blend_base} --alphas {chosen_alpha}") == 0
else:
    raise ValueError(f"unknown MODE={MODE}")

metrics = json.loads((SOUP_OUT / "metrics.json").read_text())
best = metrics["best_metrics"]
print(f"\nchosen α={chosen_alpha}")
print(json.dumps(best, indent=2))
print(f"\nvs LB submit α=0.05 (0.5527): gray {best['gray_full']:.4f} vs 0.5502")
print(f"vs ref 04 α=0.10:            gray {best['gray_full']:.4f} vs 0.5547 (reference v2)")

In [ ]:
if RUN_SUBMIT:
    assert run_cmd(f"python {BUILD_SUBMIT} --run-dir {SOUP_OUT}") == 0
    zip_path = SOUP_OUT / "matching-bge-human-ft-submit.zip"
    print("submit:", masked(zip_path), f"{zip_path.stat().st_size / 1024**2:.1f} MB")
    meta = SOUP_OUT / "submit_meta.json"
    meta.write_text(json.dumps({
        "alpha": chosen_alpha,
        "lb_prior_alpha005": 0.5526942615,
        "offline": best,
        "use_reference_v2": USE_REFERENCE_V2,
        "symmetry_tta": True,
    }, indent=2), encoding="utf-8")
    print("meta:", masked(meta))
else:
    print("RUN_SUBMIT=False")

In [ ]:
import zipfile, tempfile

zip_path = SOUP_OUT / "matching-bge-human-ft-submit.zip"
assert zip_path.exists()
with tempfile.TemporaryDirectory() as td:
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(td)
    root = next(Path(td).glob("matching-bge-human-ft*"))
    sys.path.insert(0, str(root))
    from src.utils import build_text, _score_pairs
    print("α=", chosen_alpha, "| TTA:", _score_pairs.__name__)
    print("build_text:", build_text("test", '{"Бренд":"X"}')[:40])
print("Готово — отправляй zip на LB")